# Logistic Regression (TensorFlow v2) + FGM Pipeline

This notebook mirrors the **NN + FGM + dual-stream consistency gate** workflow, but replaces the neural network with a **TensorFlow v2 logistic regression classifier**.

Pipeline steps:

1. Train a **clean logistic regression** model on the clean training split  
2. Evaluate the clean model on clean test data  
3. Generate **FGM adversarial** train and test samples  
4. Retrain a second logistic regression model using ART's **AdversarialTrainer**  
5. Evaluate both models on:
   - clean test data
   - adversarial test data
   - combined clean + adversarial test data
6. Apply the same **dual-stream consistency gate** used in the CNN / LSTM / Transformer / NN notebooks

Logistic regression here is implemented as a **single dense softmax layer** in TensorFlow v2.


In [24]:
# If needed, install once:
# !pip install tensorflow adversarial-robustness-toolbox scikit-learn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [25]:
# -----------------------------
# Configuration
# -----------------------------
DATA_PATH = Path(r"../../CSVs/dataset.csv")
LABEL_COL = "anomaly"

# Drop non-feature columns if needed
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAIN = 0.20

# Logistic regression hyperparameters
# These are stronger than the old values, which tended to underfit and predict all 0s.
LR = 5e-3
L2_REG = 1e-5
BATCH_SIZE = 64
NB_EPOCHS = 75

# FGM settings
FGM_EPS = 0.10

# Adversarial training settings
ADV_RATIO = 0.55

# Prediction threshold tuning
# The notebook tunes the threshold on validation data, but these are the search limits.
THRESHOLD_GRID = np.round(np.arange(0.10, 0.91, 0.01), 2)
DEFAULT_THRESHOLD = 0.35

# Dual-stream settings requested for this project
CONFIDENCE_THRESHOLD = 0.60
DISAGREEMENT_THRESHOLD = 0.55

# Save paths
SAVE_MODELS = True
ARTIFACT_DIR = Path(r"Artifacts\LogisticRegression")
RESULTS_DIR = Path(r"Results\LogisticRegressionResults")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [26]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()

    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )

    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=VAL_SIZE_FROM_TRAIN,
        random_state=SEED,
        stratify=y_train_full,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Full label dist={np.bincount(y, minlength=2)}")
    print(f"Train={X_train.shape}, Label dist={np.bincount(y_train, minlength=2)}")
    print(f"Val={X_val.shape}, Label dist={np.bincount(y_val, minlength=2)}")
    print(f"Test={X_test.shape}, Label dist={np.bincount(y_test, minlength=2)}")

    return (
        X_train,
        X_val,
        X_test,
        y_train.astype(np.int64),
        y_val.astype(np.int64),
        y_test.astype(np.int64),
        scaler,
        feature_cols,
    )


def to_one_hot(y: np.ndarray, num_classes: int = 2):
    return tf.keras.utils.to_categorical(y, num_classes=num_classes).astype(np.float32)


if not DATA_PATH.exists():
    raise ValueError(f"Set DATA_PATH first. Current value does not exist: {DATA_PATH}")

X_train, X_val, X_test, y_train, y_val, y_test, scaler, feature_cols = load_and_prepare(str(DATA_PATH))

y_train_oh = to_one_hot(y_train, 2)
y_val_oh = to_one_hot(y_val, 2)
y_test_oh = to_one_hot(y_test, 2)

# Balanced class weights. This is the main fix for the all-zero prediction problem.
CLASS_WEIGHTS = np.array([
    len(y_train) / (2.0 * np.sum(y_train == 0)),
    len(y_train) / (2.0 * np.sum(y_train == 1)),
], dtype=np.float32)

print("Class weights [class 0, class 1]:", CLASS_WEIGHTS)


Loaded: ..\..\CSVs\dataset.csv
Rows=2123, Features=18, Full label dist=[1689  434]
Train=(1358, 18), Label dist=[1080  278]
Val=(340, 18), Label dist=[271  69]
Test=(425, 18), Label dist=[338  87]
Class weights [class 0, class 1]: [0.6287037 2.442446 ]


In [27]:
def build_logistic_model(d_in: int, l2_reg: float = L2_REG):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(d_in,)),
        tf.keras.layers.Dense(
            2,
            activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
            name="logistic_output",
        ),
    ])


def make_art_classifier(d_in: int, lr: float = LR, l2_reg: float = L2_REG):
    model = build_logistic_model(d_in=d_in, l2_reg=l2_reg)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    # ART needs a normal loss_object for attacks. The train_step below applies class weights.
    loss_object = tf.keras.losses.CategoricalCrossentropy(
        reduction=tf.keras.losses.Reduction.NONE
    )
    class_weights_tf = tf.constant(CLASS_WEIGHTS, dtype=tf.float32)

    @tf.function
    def train_step(model_instance, x_batch, y_batch):
        with tf.GradientTape() as tape:
            predictions = model_instance(x_batch, training=True)

            per_sample_loss = loss_object(y_batch, predictions)
            sample_weights = tf.reduce_sum(y_batch * class_weights_tf, axis=1)
            loss = tf.reduce_mean(per_sample_loss * sample_weights)

            if model_instance.losses:
                loss += tf.add_n(model_instance.losses)

        gradients = tape.gradient(loss, model_instance.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model_instance.trainable_variables))

    classifier = TensorFlowV2Classifier(
        model=model,
        nb_classes=2,
        input_shape=(d_in,),
        loss_object=tf.keras.losses.CategoricalCrossentropy(),
        train_step=train_step,
    )
    return classifier


def predict_probs(art_clf, X: np.ndarray):
    return art_clf.predict(X).astype(np.float32)


def predict_labels(art_clf, X: np.ndarray, threshold: float = DEFAULT_THRESHOLD):
    probs = predict_probs(art_clf, X)
    return (probs[:, 1] >= threshold).astype(int)


def tune_threshold(art_clf, X: np.ndarray, y_true: np.ndarray, name: str):
    probs = predict_probs(art_clf, X)
    rows = []

    for threshold in THRESHOLD_GRID:
        preds = (probs[:, 1] >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "accuracy": float(accuracy_score(y_true, preds)),
            "f1": float(f1_score(y_true, preds, zero_division=0)),
            "pred_0": int(np.sum(preds == 0)),
            "pred_1": int(np.sum(preds == 1)),
        })

    df = pd.DataFrame(rows)
    best_idx = df.sort_values(["f1", "accuracy"], ascending=False).index[0]
    best = df.loc[best_idx]
    best_threshold = float(best["threshold"])

    print(f"Best threshold for {name}: {best_threshold:.2f}")
    print(best.to_dict())

    return best_threshold, df


def eval_classifier(art_clf, X: np.ndarray, y_true: np.ndarray, name: str, threshold: float):
    y_pred = predict_labels(art_clf, X, threshold=threshold)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"[{name}] threshold={threshold:.2f} acc={acc:.4f} f1={f1:.4f}")
    print("y_true counts:", np.bincount(y_true, minlength=2))
    print("y_pred counts:", np.bincount(y_pred, minlength=2))
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    metrics = {
        "model_eval": name,
        "threshold": float(threshold),
        "acc": float(acc),
        "f1": float(f1),
        "pred_0": int(np.sum(y_pred == 0)),
        "pred_1": int(np.sum(y_pred == 1)),
    }
    return metrics, y_pred


def save_art_model_state(art_clf, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    art_clf.model.save_weights(str(out_path))
    print(f"Saved model weights: {out_path}")


In [28]:
# Step 1: Train clean logistic regression model
print("Training clean TensorFlow v2 logistic regression with:", {
    "learning_rate": LR,
    "l2_reg": L2_REG,
    "batch_size": BATCH_SIZE,
    "epochs": NB_EPOCHS,
    "fgm_eps": FGM_EPS,
    "adv_ratio": ADV_RATIO,
    "class_weights": CLASS_WEIGHTS.tolist(),
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train_oh, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

clean_threshold, clean_threshold_df = tune_threshold(
    art_clean,
    X_val,
    y_val,
    "clean_model_on_validation",
)

clean_threshold_df.to_csv(RESULTS_DIR / "logreg_tf2_clean_threshold_search.csv", index=False)

clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
    threshold=clean_threshold,
)

if SAVE_MODELS:
    save_art_model_state(art_clean, ARTIFACT_DIR / "logreg_tf2_clean.weights.h5")


Training clean TensorFlow v2 logistic regression with: {'learning_rate': 0.005, 'l2_reg': 1e-05, 'batch_size': 64, 'epochs': 75, 'fgm_eps': 0.1, 'adv_ratio': 0.55, 'class_weights': [0.6287037134170532, 2.442445993423462]}
Best threshold for clean_model_on_validation: 0.43
{'threshold': 0.43, 'accuracy': 0.9294117647058824, 'f1': 0.8208955223880597, 'pred_0': 275.0, 'pred_1': 65.0}
[clean_model_on_clean_test] threshold=0.43 acc=0.9106 f1=0.7912
y_true counts: [338  87]
y_pred counts: [330  95]
confusion matrix:
[[315  23]
 [ 15  72]]
              precision    recall  f1-score   support

           0     0.9545    0.9320    0.9431       338
           1     0.7579    0.8276    0.7912        87

    accuracy                         0.9106       425
   macro avg     0.8562    0.8798    0.8672       425
weighted avg     0.9143    0.9106    0.9120       425

Saved model weights: Artifacts\LogisticRegression\logreg_tf2_clean.weights.h5


In [29]:
# Step 2: Generate FGM adversarial samples from the clean model
fgm_clean = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)

X_train_adv = fgm_clean.generate(x=X_train).astype(np.float32)
X_val_adv = fgm_clean.generate(x=X_val).astype(np.float32)
X_test_adv = fgm_clean.generate(x=X_test).astype(np.float32)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_val_adv:", X_val_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# Ground-truth labels stay aligned with the original data
y_train_adv = y_train.copy()
y_val_adv = y_val.copy()
y_test_adv = y_test.copy()
y_train_adv_oh = to_one_hot(y_train_adv, 2)
y_val_adv_oh = to_one_hot(y_val_adv, 2)
y_test_adv_oh = to_one_hot(y_test_adv, 2)

# Combined clean + adversarial test set
X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)


Adversarial data generated:
X_train_adv: (1358, 18)
X_val_adv: (340, 18)
X_test_adv: (425, 18)


In [30]:
# Evaluate clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
    threshold=clean_threshold,
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
    threshold=clean_threshold,
)


[clean_model_on_adv_test] threshold=0.43 acc=0.0965 f1=0.1111
y_true counts: [338  87]
y_pred counts: [ 80 345]
confusion matrix:
[[ 17 321]
 [ 63  24]]
              precision    recall  f1-score   support

           0     0.2125    0.0503    0.0813       338
           1     0.0696    0.2759    0.1111        87

    accuracy                         0.0965       425
   macro avg     0.1410    0.1631    0.0962       425
weighted avg     0.1832    0.0965    0.0874       425

[clean_model_on_combined_test] threshold=0.43 acc=0.5035 f1=0.3127
y_true counts: [676 174]
y_pred counts: [410 440]
confusion matrix:
[[332 344]
 [ 78  96]]
              precision    recall  f1-score   support

           0     0.8098    0.4911    0.6114       676
           1     0.2182    0.5517    0.3127       174

    accuracy                         0.5035       850
   macro avg     0.5140    0.5214    0.4621       850
weighted avg     0.6887    0.5035    0.5503       850



In [31]:
# Step 3: Adversarial training with ART's AdversarialTrainer
# Important fix: the training attack must use art_adv as its estimator, not art_clean.
art_adv = make_art_classifier(d_in=X_train.shape[1])

fgm_for_training = FastGradientMethod(estimator=art_adv, eps=FGM_EPS)

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=fgm_for_training,
    ratio=ADV_RATIO,
)

adv_trainer.fit(
    X_train,
    y_train_oh,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

adv_threshold, adv_threshold_df = tune_threshold(
    art_adv,
    X_val,
    y_val,
    "adv_trained_model_on_validation",
)

adv_threshold_df.to_csv(RESULTS_DIR / "logreg_tf2_adv_threshold_search.csv", index=False)

if SAVE_MODELS:
    save_art_model_state(art_adv, ARTIFACT_DIR / "logreg_tf2_adversarial_trained.weights.h5")


Adversarial training epochs: 100%|██████████| 75/75 [00:54<00:00,  1.36it/s]

Best threshold for adv_trained_model_on_validation: 0.50
{'threshold': 0.5, 'accuracy': 0.8, 'f1': 0.43333333333333335, 'pred_0': 289.0, 'pred_1': 51.0}
Saved model weights: Artifacts\LogisticRegression\logreg_tf2_adversarial_trained.weights.h5


In [32]:
# Step 4: Evaluate adversarially trained model
adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
    threshold=adv_threshold,
)

adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
    threshold=adv_threshold,
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
    threshold=adv_threshold,
)


[adv_trained_model_on_clean_test] threshold=0.50 acc=0.7835 f1=0.3784
y_true counts: [338  87]
y_pred counts: [364  61]
confusion matrix:
[[305  33]
 [ 59  28]]
              precision    recall  f1-score   support

           0     0.8379    0.9024    0.8689       338
           1     0.4590    0.3218    0.3784        87

    accuracy                         0.7835       425
   macro avg     0.6485    0.6121    0.6237       425
weighted avg     0.7603    0.7835    0.7685       425

[adv_trained_model_on_adv_test] threshold=0.50 acc=0.6871 f1=0.2888
y_true counts: [338  87]
y_pred counts: [325 100]
confusion matrix:
[[265  73]
 [ 60  27]]
              precision    recall  f1-score   support

           0     0.8154    0.7840    0.7994       338
           1     0.2700    0.3103    0.2888        87

    accuracy                         0.6871       425
   macro avg     0.5427    0.5472    0.5441       425
weighted avg     0.7037    0.6871    0.6949       425

[adv_trained_model_on_comb

In [33]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

display(summary_df)

summary_path = RESULTS_DIR / "logreg_tf2_fgm_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")


,model_eval,threshold,acc,f1,pred_0,pred_1
0,clean_model_on_clean_test,0.43,0.910588,0.791209,330,95
1,clean_model_on_adv_test,0.43,0.096471,0.111111,80,345
2,clean_model_on_combined_test,0.43,0.503529,0.312704,410,440
3,adv_trained_model_on_clean_test,0.50,0.783529,0.378378,364,61
4,adv_trained_model_on_adv_test,0.50,0.687059,0.288770,325,100
5,adv_trained_model_on_combined_test,0.50,0.735294,0.328358,689,161


Saved: Results\LogisticRegressionResults\logreg_tf2_fgm_pipeline_summary.csv


## Dual-Stream Consistency Gate

This section keeps the same overall dual-stream idea:

- Nominal model = clean logistic regression model
- Guardian model = adversarially trained logistic regression model
- Attack flags come from prediction disagreement and anomaly-probability disagreement

This fixed version uses the tuned anomaly thresholds instead of raw `argmax`, because raw `argmax` was part of why the old notebook kept predicting only class 0.


In [34]:
def predict_with_confidence(art_clf, X: np.ndarray, threshold: float):
    probs = predict_probs(art_clf, X)
    preds = (probs[:, 1] >= threshold).astype(int)
    pred_conf = np.where(preds == 1, probs[:, 1], probs[:, 0])
    return preds, probs, pred_conf


class DualStreamDetector:
    def __init__(self, nominal_model, guardian_model, nominal_threshold: float, guardian_threshold: float):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model
        self.nominal_threshold = nominal_threshold
        self.guardian_threshold = guardian_threshold

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal, confs_nominal = predict_with_confidence(
            self.nominal_model, X, self.nominal_threshold
        )
        preds_guardian, probs_guardian, confs_guardian = predict_with_confidence(
            self.guardian_model, X, self.guardian_threshold
        )

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(confs_nominal[i])
            conf_guardian = float(confs_guardian[i])
            prob_diff = float(pB[1] - pA[1])

            detected = False
            reasons = []

            if yA != yB:
                detected = True
                reasons.append("disagreement")

            if yA == 0 and yB == 1:
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df


def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("y_true counts:", np.bincount(y_true, minlength=2))
    print("y_pred counts:", np.bincount(y_pred, minlength=2))
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": float(acc),
        "f1": float(f1),
        "pred_0": int(np.sum(y_pred == 0)),
        "pred_1": int(np.sum(y_pred == 1)),
    }


detector = DualStreamDetector(
    nominal_model=art_clean,
    guardian_model=art_adv,
    nominal_threshold=clean_threshold,
    guardian_threshold=adv_threshold,
)

gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("Dual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "FGM",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "clean_threshold": float(clean_threshold),
    "guardian_threshold": float(adv_threshold),
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("Dual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("Gate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("Gate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))


[dual_stream_final_predictions_on_clean_test] acc=0.7835 f1=0.3784
y_true counts: [338  87]
y_pred counts: [364  61]
confusion matrix:
[[305  33]
 [ 59  28]]
              precision    recall  f1-score   support

           0     0.8379    0.9024    0.8689       338
           1     0.4590    0.3218    0.3784        87

    accuracy                         0.7835       425
   macro avg     0.6485    0.6121    0.6237       425
weighted avg     0.7603    0.7835    0.7685       425

[dual_stream_final_predictions_on_adv_test] acc=0.6871 f1=0.2888
y_true counts: [338  87]
y_pred counts: [325 100]
confusion matrix:
[[265  73]
 [ 60  27]]
              precision    recall  f1-score   support

           0     0.8154    0.7840    0.7994       338
           1     0.2700    0.3103    0.2888        87

    accuracy                         0.6871       425
   macro avg     0.5427    0.5472    0.5441       425
weighted avg     0.7037    0.6871    0.6949       425

[dual_stream_final_predictions_o

,model_eval,acc,f1,pred_0,pred_1
0,dual_stream_final_predictions_on_clean_test,0.783529,0.378378,364,61
1,dual_stream_final_predictions_on_adv_test,0.687059,0.288770,325,100
2,dual_stream_final_predictions_on_combined_test,0.735294,0.328358,689,161


Dual-stream attack-detection summary:


,attack,confidence_threshold,disagreement_threshold,clean_threshold,guardian_threshold,FPR,TPR,F1,clean_model_acc_on_adv,guardian_model_acc_on_clean,guardian_model_acc_on_adv
0,FGM,0.6,0.55,0.43,0.5,0.183529,0.703529,0.745636,0.096471,0.783529,0.687059


Gate reason counts on clean test:


,reason,count
0,none,347
1,disagreement,78


Gate reason counts on adversarial test:


,reason,count
0,disagreement,299
1,none,126


In [ ]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "logreg_tf2_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "logreg_tf2_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")


Saved: Results\LogisticRegressionResults\logreg_tf2_dual_stream_prediction_summary.csv
Saved: Results\LogisticRegressionResults\logreg_tf2_dual_stream_detection_results.csv


: 

<!-- ## What to tweak if F1 is still weak

Start with these settings:

- `THRESHOLD_GRID`, especially lower values like 0.15 to 0.40
- `FGM_EPS`, try 0.05, 0.10, 0.15
- `ADV_RATIO`, try 0.40, 0.55, 0.70
- `NB_EPOCHS`, try 100 if training still looks unstable
- `LR`, try 1e-3 if 5e-3 is too jumpy

The most important sanity check is `y_pred counts`. If it says `[425, 0]` again, the model is still predicting only normal traffic and you need either stronger class weighting, a lower threshold, or a stronger model than logistic regression. -->
